In [ ]:
import os

# The variable 'zip_path' was assigned the path to a directory, not a zip file.
# Attempting to open a directory with `zipfile.ZipFile` causes an `IsADirectoryError`.
# Assuming '/content/balanced_realwaste' is the directory that contains your data,
# we can directly work with it.

zip_path = "/content/balanced_realwaste"

# If the intention was to list the contents of this directory:
print(os.listdir(zip_path))

['Food Organics', 'Plastic', 'Cardboard', 'Textile Trash', 'Glass', 'Miscellaneous Trash', 'Vegetation', 'Paper', 'Metal']


In [ ]:
import os

path = "/content/balanced_realwaste"
print(os.listdir(path))

['Food Organics', 'Plastic', 'Cardboard', 'Textile Trash', 'Glass', 'Miscellaneous Trash', 'Vegetation', 'Paper', 'Metal']


In [ ]:
data_dir = path

In [ ]:
classes = os.listdir(data_dir)
print("Classes:",classes)
print("Total Classes:",len(classes))

Classes: ['Food Organics', 'Plastic', 'Cardboard', 'Textile Trash', 'Glass', 'Miscellaneous Trash', 'Vegetation', 'Paper', 'Metal']
Total Classes: 9


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
from tensorflow.keras.applications.mobilenet import preprocess_input

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # MobileNet scaling
    validation_split=0.3,   # 70% train / 30% validation
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

In [ ]:
train_data = train_datagen.flow_from_directory(
    data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

Found 2007 images belonging to 9 classes.


In [ ]:
val_data = train_datagen.flow_from_directory(
    data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

Found 855 images belonging to 9 classes.


In [ ]:
print("Classes:", train_data.class_indices)

Classes: {'Cardboard': 0, 'Food Organics': 1, 'Glass': 2, 'Metal': 3, 'Miscellaneous Trash': 4, 'Paper': 5, 'Plastic': 6, 'Textile Trash': 7, 'Vegetation': 8}


In [ ]:
from tensorflow.keras.applications.mobilenet import MobileNet

base_model = MobileNet(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

17225924/17225924 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
for layer in base_model.layers:
    layer.trainable = False

In [ ]:
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)

output = Dense(9, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    callbacks=[early_stop]
)

Epoch 1/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 188s 3s/step - accuracy: 0.5820 - loss: 1.2201 - val_accuracy: 0.5731 - val_loss: 1.2166
Epoch 2/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 168s 3s/step - accuracy: 0.7897 - loss: 0.6075 - val_accuracy: 0.6409 - val_loss: 1.1235
Epoch 3/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 166s 3s/step - accuracy: 0.8590 - loss: 0.4330 - val_accuracy: 0.6643 - val_loss: 1.0452
Epoch 4/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 166s 3s/step - accuracy: 0.8859 - loss: 0.3470 - val_accuracy: 0.6737 - val_loss: 1.0377
Epoch 5/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 166s 3s/step - accuracy: 0.9123 - loss: 0.2828 - val_accuracy: 0.6573 - val_loss: 1.0879
Epoch 6/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 199s 3s/step - accuracy: 0.9158 - loss: 0.2495 - val_accuracy: 0.6702 - val_loss: 1.1044


In [ ]:
model.save("9_waste_classifier_model.h5")

In [ ]:
from google.colab import files
files.download("9_waste_classifier_model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json

with open("9_classes.json", "w") as f:
    json.dump(train_data.class_indices, f)

files.download("9_classes.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np
from tensorflow.keras.applications.mobilenet import preprocess_input

img_path = "/content/tin can waste.jpg"

img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

img_array = preprocess_input(img_array)

prediction = model.predict(img_array)

classes = list(train_data.class_indices.keys())
print("Prediction:", classes[np.argmax(prediction)])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
Prediction: Metal
